# 01-03 ReLU 函数公式推导

ReLU 是现代深度学习中最常用的隐藏层激活函数之一。它的公式非常简单，但背后的作用很重要：保留正信号，截断负信号。

## 1. 为什么需要 ReLU

Sigmoid 和 Tanh 都会把输入压缩到有限区间。当 $z$ 很大或很小时，它们的曲线会变平，导数接近 $0$，导致梯度传递变弱。

ReLU 的想法很直接：

- 如果输入是正数，就原样通过。
- 如果输入不是正数，就输出 $0$。

这样既引入了非线性，又让正半轴的梯度保持为 $1$。

## 2. 从阈值函数到 ReLU

早期感知机使用阶跃函数：

$$
\operatorname{step}(z)=
\begin{cases}
1, & z>0 \\
0, & z\le 0
\end{cases}
$$

阶跃函数能做分类，但它几乎处处导数为 $0$，不适合梯度下降训练。

ReLU 可以看成更适合优化的阈值函数：

$$
\operatorname{ReLU}(z)=
\begin{cases}
z, & z>0 \\
0, & z\le 0
\end{cases}
$$

它也可以写成更简洁的形式：

$$
\operatorname{ReLU}(z)=\max(0,z)
$$

## 3. 公式为什么是 $\max(0,z)$

因为 ReLU 的规则就是从 $0$ 和 $z$ 里面选更大的那个。

当 $z>0$ 时：

$$
\max(0,z)=z
$$

当 $z\le 0$ 时：

$$
\max(0,z)=0
$$

所以：

$$
\max(0,z)=
\begin{cases}
z, & z>0 \\
0, & z\le 0
\end{cases}
$$

这就是 ReLU 的完整定义。

## 4. 导数推导

ReLU 是分段函数，所以导数也分段讨论。

当 $z>0$ 时：

$$
\operatorname{ReLU}(z)=z
$$

因此：

$$
\operatorname{ReLU}'(z)=1
$$

当 $z<0$ 时：

$$
\operatorname{ReLU}(z)=0
$$

因此：

$$
\operatorname{ReLU}'(z)=0
$$

所以：

$$
\operatorname{ReLU}'(z)=
\begin{cases}
1, & z>0 \\
0, & z<0
\end{cases}
$$

在 $z=0$ 处，左导数为 $0$，右导数为 $1$，所以严格来说不可导。深度学习框架通常会人为约定一个值，例如 $0$，这不影响实际训练。

## 5. 为什么 ReLU 适合深层网络

ReLU 的正半轴导数是：

$$
\operatorname{ReLU}'(z)=1, \quad z>0
$$

这意味着只要神经元处于激活状态，梯度可以比较顺畅地向前一层传递。

相比之下，Sigmoid 和 Tanh 在输入绝对值很大时会进入饱和区：

$$
\sigma'(z)\approx 0
$$

$$
\tanh'(z)\approx 0
$$

因此 ReLU 常常让深层网络训练得更快、更稳定。

## 6. 死亡 ReLU 问题

ReLU 也有缺点。如果某个神经元长期满足：

$$
z\le 0
$$

那么它的输出就是：

$$
\operatorname{ReLU}(z)=0
$$

对应梯度也是：

$$
\operatorname{ReLU}'(z)=0
$$

如果这种情况一直发生，这个神经元参数很难再更新，相当于“不工作”了。这就是死亡 ReLU 问题。

常见缓解方式包括：使用较合适的学习率、合理初始化权重，或者使用 Leaky ReLU 等变体。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-5, 5, 500)
relu = np.maximum(0, z)
relu_grad = np.where(z > 0, 1, 0)
leaky_relu = np.where(z > 0, z, 0.1 * z)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z, relu, label='ReLU')
axes[0].plot(z, leaky_relu, linestyle='--', label='Leaky ReLU')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('ReLU and Leaky ReLU')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(z, relu_grad, color='#DC2626', label="ReLU derivative")
axes[1].set_ylim(-0.1, 1.1)
axes[1].set_title('Derivative of ReLU')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. 在神经网络中的使用

普通前馈神经网络的隐藏层经常写成：

$$
\mathbf{h}=\operatorname{ReLU}(\mathbf{W}\mathbf{x}+\mathbf{b})
$$

在 PyTorch 中对应：

```python
nn.Linear(in_features, out_features)
nn.ReLU()
```

先记住一个实用规则：如果是普通隐藏层，不知道选什么激活函数时，优先尝试 ReLU。